In [5]:
!pwd

/home/jupyter/mlops/GA_notebooks


In [4]:
#Check data and model existing..else dvc pull
!ls -lart ../data.csv ../model.joblib

-rw-r--r-- 1 jupyter jupyter   2606 Oct 16 08:12 ../data.csv
-rw-r--r-- 1 jupyter jupyter 150417 Oct 16 08:12 ../model.joblib


# Get the cluster ready

In [6]:
!gcloud container clusters update iris-cluster \
    --enable-autoscaling \
    --min-nodes 1 \
    --max-nodes 3 \
    --zone us-central1-a

Updating iris-cluster...done.                                                  
Updated [https://container.googleapis.com/v1/projects/second-capsule-472911-g7/zones/us-central1-a/clusters/iris-cluster].
To inspect the contents of your cluster, go to: https://console.cloud.google.com/kubernetes/workload_/gcloud/us-central1-a/iris-cluster?project=second-capsule-472911-g7


# Add autoscaling

In [8]:
%%writefile ../k8s/hpa.yaml
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: iris-api-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: iris-api
  minReplicas: 1
  maxReplicas: 3
  metrics:
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: 60


Writing ../k8s/hpa.yaml


# Add resource limits to deployment.yaml

In [13]:
%%writefile ../k8s/deployment.yaml
spec:
  template:
    spec:
      containers:
        - name: iris-api
          image: us-central1-docker.pkg.dev/second-capsule-472911-g7/iris-repo/iris-api:latest
          ports:
            - containerPort: 8080
          resources:
            requests:
              cpu: "200m"
              memory: "256Mi"
            limits:
              cpu: "500m"
              memory: "512Mi"


Overwriting ../k8s/deployment.yaml


# Create file wrk/post.lua in repo root

In [10]:
!mkdir -p ../wrk

In [11]:
%%writefile ../wrk/post.lua
wrk.method = "POST"
wrk.body   = '{"sepal_length":5.1,"sepal_width":3.5,"petal_length":1.4,"petal_width":0.2}'
wrk.headers["Content-Type"] = "application/json"

Writing ../wrk/post.lua


In [15]:
%%writefile ../.github/workflows/ci.yml
name: CI-CD

permissions:
  contents: write
  pull-requests: write
  id-token: write

on:
  push:
  pull_request:

env:
  PROJECT_ID: ${{ secrets.GCP_PROJECT }}
  ARTIFACT_REGION: ${{ secrets.ARTIFACT_REGION }}
  ARTIFACT_REPO: ${{ secrets.ARTIFACT_REPO }}
  GKE_CLUSTER: ${{ secrets.GKE_CLUSTER }}
  GKE_ZONE: ${{ secrets.GKE_ZONE }}

jobs:
  # ---------------------
  # Test Job
  # ---------------------
  test:
    runs-on: ubuntu-latest
    env:
      MLFLOW_TRACKING_URI: ${{ secrets.MLFLOW_TRACKING_URI }}

    steps:
      - uses: actions/checkout@v3

      - name: Setup Python
        uses: actions/setup-python@v4
        with:
          python-version: '3.10'

      - uses: iterative/setup-dvc@v1
        with:
          version: 3.63.0

      - name: Install dependencies
        run: |
          pip install pytest pandas joblib scikit-learn mlflow

      - name: Authenticate GCP
        env:
          GCP_SA_KEY: ${{ secrets.GCP_SA_KEY }}
          PROJECT_ID: ${{ secrets.PROJECT_ID }}
        run: |
          echo "$GCP_SA_KEY" > /tmp/key.json
          gcloud auth activate-service-account --key-file=/tmp/key.json
          gcloud config set project $PROJECT_ID
          export GOOGLE_APPLICATION_CREDENTIALS=/tmp/key.json

      - name: Pull data via DVC
        run: |
          export GOOGLE_APPLICATION_CREDENTIALS=/tmp/key.json
          dvc pull -r mygcs --force

      - name: Run tests
        run: pytest -v -s tests/test_pipeline_mlflow.py --uri "$MLFLOW_TRACKING_URI" > report.txt

      - name: Set up CML
        uses: iterative/setup-cml@v2

      - name: Post CML comment
        env:
          REPO_TOKEN: ${{ secrets.GITHUB_TOKEN }}
        run: |
          echo "### Pytest Report" > report.md
          cat report.txt >> report.md
          cml comment create report.md


  # ---------------------
  # Build Docker image and push to Artifact Registry
  # ---------------------
  build:
    runs-on: ubuntu-latest
    needs: test
    steps:
      - uses: actions/checkout@v3
    
      - uses: iterative/setup-dvc@v1
        with:
          version: 3.63.0

      - name: Authenticate GCP
        env:
          GCP_SA_KEY: ${{ secrets.GCP_SA_KEY }}
          PROJECT_ID: ${{ secrets.PROJECT_ID }}
        run: |
          echo "$GCP_SA_KEY" > /tmp/key.json
          gcloud auth activate-service-account --key-file=/tmp/key.json
          gcloud config set project $PROJECT_ID
          gcloud auth configure-docker ${{ secrets.ARTIFACT_REGION }}-docker.pkg.dev --quiet
        
      - name: Pull data via DVC
        run: |
          export GOOGLE_APPLICATION_CREDENTIALS=/tmp/key.json
          dvc pull -r mygcs --force

      - name: Build and Push Docker image
        env:
          ARTIFACT_REGION: ${{ secrets.ARTIFACT_REGION }}
          PROJECT_ID: ${{ secrets.PROJECT_ID }}
          ARTIFACT_REPO: ${{ secrets.ARTIFACT_REPO }}
          IMAGE_NAME: iris-api
        run: |
          IMAGE_URI=${ARTIFACT_REGION}-docker.pkg.dev/${PROJECT_ID}/${ARTIFACT_REPO}/${IMAGE_NAME}:latest
          docker build -t $IMAGE_URI .
          docker push $IMAGE_URI
          echo "IMAGE_URI=$IMAGE_URI" >> $GITHUB_ENV


  # ---------------------
  # Deploy to GKE
  # ---------------------
  deploy:
      runs-on: ubuntu-latest
      needs: build

      permissions:
        contents: write
        pull-requests: write

      steps:
        - uses: actions/checkout@v3

        - name: Authenticate to Google Cloud
          uses: google-github-actions/auth@v2
          with:
            credentials_json: ${{ secrets.GCP_SA_KEY }}

        - name: Connect to GKE cluster
          uses: google-github-actions/get-gke-credentials@v2
          with:
            cluster_name: ${{ secrets.GKE_CLUSTER }}
            location: ${{ secrets.GKE_ZONE }}
            project_id: ${{ secrets.PROJECT_ID }}

        - name: Apply Deployment + HPA
          env:
            IMAGE_URI: ${{ secrets.ARTIFACT_REGION }}-docker.pkg.dev/${{ secrets.PROJECT_ID }}/${{ secrets.ARTIFACT_REPO }}/iris-api:latest
          run: |
            kubectl apply -f k8s/deployment.yaml
            kubectl set image deployment/iris-api iris-api=$IMAGE_URI
            kubectl apply -f k8s/hpa.yaml
            kubectl rollout status deployment/iris-api

        - name: Install wrk
          run: |
            sudo apt-get update
            sudo apt-get install -y wrk

        - name: Get LoadBalancer IP
          id: get_ip
          run: |
            IP=$(kubectl get svc iris-api-svc --output jsonpath='{.status.loadBalancer.ingress[0].ip}')
            echo "SERVICE_IP=$IP" >> $GITHUB_ENV

        - name: Load Test (autoscaling enabled)
          run: |
            echo "### Test: HPA enabled (maxPods=3)" > wrk_report.md
            wrk -t12 -c1000 -d20s http://$SERVICE_IP/predict --script=wrk/post.lua >> wrk_report.md
            echo "\nPods after test:" >> wrk_report.md
            kubectl get pods >> wrk_report.md
            echo "\nHPA status:" >> wrk_report.md
            kubectl get hpa >> wrk_report.md

        - name: Restrict scaling to 1 pod and test again
          run: |
            kubectl scale deployment iris-api --replicas=1
            kubectl delete hpa iris-api-hpa --ignore-not-found
            echo "### Test: Forced single pod (maxPods=1)" >> wrk_report.md
            wrk -t12 -c2000 -d20s http://$SERVICE_IP/predict --script=wrk/post.lua >> wrk_report.md
            echo "\nPods after forced single pod test:" >> wrk_report.md
            kubectl get pods >> wrk_report.md

        - name: Post Load Test Results to PR
          uses: iterative/setup-cml@v2
          env:
            REPO_TOKEN: ${{ secrets.GITHUB_TOKEN }}
          run: |
            cml comment create wrk_report.md


Overwriting ../.github/workflows/ci.yml


In [ ]:
!git add .
!git commit -m "CD pipeline update"